# 01 — Data Overview

This notebook describes the **Ausgrid Solar Home Electricity dataset** used in the paper and visualises the preprocessed input files provided in `data/`.

The goal is to give an intuition of what the model receives as input, without requiring access to the original raw CSV.

> **Paper:** *Conditional diffusion modeling for probabilistic behind-the-meter PV disaggregation*  
> Jené-Vinuesa et al., *Energy and AI*, 2026 · [DOI](https://doi.org/10.1016/j.egyai.2026.100692)


## 1. Dataset description

The **Ausgrid** dataset contains one full year of half-hourly smart meter readings (July 2012 – June 2013) from **300 residential customers** in New South Wales, Australia. Each customer has:

- A **net consumption (NC)** profile: the algebraic sum of load and PV generation as seen by the meter
- A **submetered PV** profile: the actual behind-the-meter PV generation (used as ground truth)
- A **postcode**: used to retrieve location-specific GHI from [Open-Meteo](https://open-meteo.com/)

After cleaning, **43 customers** were excluded due to anomalous or incomplete data. The remaining **257 customers** are split into:
- **107 training customers** — used to learn the PV generation distribution
- **150 test customers** — held out for evaluation

Each daily profile has **T = 48 time steps** (one per 30 minutes).


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Add src/ to path so we can reuse utils
sys.path.insert(0, "../src")

DATA_DIR = "../data"

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 110,
})


## 2. Load preprocessed data

In [ ]:
pv   = pd.read_csv(f"{DATA_DIR}/pv_norm_cap_v2.csv",       index_col=0)
gi   = pd.read_csv(f"{DATA_DIR}/ghi_norm_v2.csv",          index_col=0)
nc   = pd.read_csv(f"{DATA_DIR}/nc_norm_maxexp_v2.csv",    index_col=0)
nc1  = pd.read_csv(f"{DATA_DIR}/nc_lag1_maxexp_v2.csv",    index_col=0)
cap  = pd.read_csv(f"{DATA_DIR}/capacity_estimation_GG_pred.csv", index_col=0)

print(f"PV profiles  : {pv.shape[0]:,} samples × {pv.shape[1]} time steps")
print(f"GHI profiles : {gi.shape[0]:,} samples × {gi.shape[1]} time steps")
print(f"NC profiles  : {nc.shape[0]:,} samples × {nc.shape[1]} time steps")
print(f"NC lag-1     : {nc1.shape[0]:,} samples × {nc1.shape[1]} time steps")
print(f"\nIndex format : <customer_id>_<DD/MM/YYYY>")
print(f"Example      : {pv.index[200]}")


In [ ]:
# Parse customer IDs and dates from the index
def parse_index(df):
    parts = df.index.str.split("_")
    df = df.copy()
    df["customer"] = parts.str[0].astype(int)
    df["date"]     = pd.to_datetime(parts.str[1], format="%d/%m/%Y")
    return df

pv_meta = parse_index(pv)
customers = sorted(pv_meta["customer"].unique())
dates     = pv_meta["date"].unique()

print(f"Unique customers : {len(customers)}")
print(f"Date range       : {min(dates).date()} → {max(dates).date()}")
print(f"Days per customer: ~{pv_meta.groupby('customer').size().mean():.0f}")


## 3. Normalisation scheme

Each signal is normalised so that values are roughly in [0, 1] for stable model training:

| Signal | Normalisation |
|--------|--------------|
| **PV** | Divided by predicted PV system capacity (`cap_pred`) |
| **GHI** | Divided by 1000 W/m² |
| **NC** | Divided by customer's maximum export value (`max_export`) |
| **NC lag-1** | Same as NC, using the previous day's readings |

The capacity estimates in `capacity_estimation_GG_pred.csv` were derived from the maximum observed export values using a regression-based approach.


In [ ]:
print("Capacity estimates (kW) — first 10 customers:")
print(cap[["cap_pred", "cap_real", "max_export"]].head(10).to_string())
print(f"\nMean real capacity : {cap['cap_real'].mean():.2f} kW")
print(f"Mean pred capacity : {cap['cap_pred'].mean():.2f} kW")


## 4. Visualise example profiles

We select a few representative days (summer and winter) for a single customer to show what the model conditions on.


In [ ]:
# Pick a customer and a summer / winter day
CUSTOMER = 289

pv_c  = pv[pv.index.str.startswith(f"{CUSTOMER}_")]
gi_c  = gi[gi.index.str.startswith(f"{CUSTOMER}_")]
nc_c  = nc[nc.index.str.startswith(f"{CUSTOMER}_")]

# Summer (January in Southern Hemisphere) and Winter (June)
summer_key = f"{CUSTOMER}_15/01/2013"
winter_key = f"{CUSTOMER}_15/06/2013"

t = np.arange(48) * 0.5          # hours (0 to 23.5)
cap_real = cap.loc[CUSTOMER, "cap_real"]

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=False)

for col, (key, season) in enumerate([(summer_key, "Summer (Jan 2013)"),
                                      (winter_key, "Winter (Jun 2013)")]):
    ax_pv = axes[0, col]
    ax_nc = axes[1, col]

    pv_kw = pv.loc[key].values * cap_real
    gi_v  = gi.loc[key].values
    nc_v  = nc.loc[key].values

    ax_pv.fill_between(t, 0, pv_kw, alpha=0.4, color="#F39C12", label="PV (kW)")
    ax_pv.plot(t, pv_kw, color="#F39C12", linewidth=1.5)
    ax_pv2 = ax_pv.twinx()
    ax_pv2.plot(t, gi_v, color="#3498DB", linewidth=1.2, linestyle="--", label="GHI (norm)")
    ax_pv2.set_ylabel("GHI (normalised)", color="#3498DB")
    ax_pv2.tick_params(axis="y", colors="#3498DB")
    ax_pv2.set_ylim(0, 1.4)

    ax_pv.set_title(f"Customer {CUSTOMER} — {season}")
    ax_pv.set_ylabel("PV generation (kW)")
    ax_pv.set_xlim(0, 23.5)
    ax_pv.set_ylim(bottom=0)

    ax_nc.plot(t, nc_v, color="#2C3E50", linewidth=1.5, label="NC (norm)")
    ax_nc.axhline(0, color="gray", linewidth=0.8, linestyle=":")
    ax_nc.fill_between(t, nc_v, 0, where=(nc_v < 0), alpha=0.3,
                        color="#E74C3C", label="Export (NC < 0)")
    ax_nc.fill_between(t, nc_v, 0, where=(nc_v >= 0), alpha=0.2,
                        color="#2ECC71", label="Import (NC ≥ 0)")
    ax_nc.set_xlabel("Hour of day")
    ax_nc.set_ylabel("Net consumption (normalised)")
    ax_nc.set_xlim(0, 23.5)
    ax_nc.legend(fontsize=9, loc="upper left")

axes[0, 0].legend(fontsize=9, loc="upper left")
fig.tight_layout()
plt.show()


## 5. Distribution of PV profiles across customers

To give a sense of variability, we plot the distribution of daily peak PV generation across all customers.


In [ ]:
# Compute daily peak for each sample (in kW, using cap_real)
peaks = []
for idx, row in pv.iterrows():
    cust = int(idx.split("_")[0])
    if cust in cap.index:
        peaks.append(row.max() * cap.loc[cust, "cap_real"])

peaks = np.array(peaks)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(peaks, bins=50, color="#F39C12", edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Daily peak PV generation (kW)")
axes[0].set_ylabel("Number of daily profiles")
axes[0].set_title("Distribution of daily peak PV output")

# Median daily profile across all customers
median_profile = pv.median(axis=0).values
q25_profile    = pv.quantile(0.25, axis=0).values
q75_profile    = pv.quantile(0.75, axis=0).values

axes[1].fill_between(t, q25_profile, q75_profile, alpha=0.35,
                     color="#F39C12", label="25–75th percentile")
axes[1].plot(t, median_profile, color="#E67E22", linewidth=2, label="Median")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Normalised PV generation")
axes[1].set_title("Median normalised PV profile (all customers × all days)")
axes[1].legend()
axes[1].set_xlim(0, 23.5)

fig.tight_layout()
plt.show()
print(f"Median daily peak: {np.median(peaks):.2f} kW | 90th percentile: {np.percentile(peaks, 90):.2f} kW")


## 6. Train / test split

The 257 customers are split into training and test sets. The split is customer-level — a customer's data is either fully in training or fully in test, never both. This tests the model's ability to generalise to unseen households.


In [ ]:
from main import DROP_CUSTOMERS, BEST_CONFIGS
from utils import split_train_test_tensors

config = BEST_CONFIGS["cddpm"]

# Re-use the same split function as the training script
all_conditions = {"gi": gi, "nc": nc, "nc_lag1": nc1}
cond_data = {k: v for k, v in all_conditions.items() if k in config["conditions"]}

exp = split_train_test_tensors(
    pv, cond_data,
    seed=config["seed"],
    test_size=config["test_size"],
    train_size=config["train_size"],
    drop_customers=DROP_CUSTOMERS,
)

print(f"Training samples : {exp['pv_train'].shape[0]:,} daily profiles "
      f"({config['train_size']} customers)")
print(f"Test samples     : {exp['pv_test'].shape[0]:,} daily profiles "
      f"({config['test_size']} customers)")
print(f"Profile length   : {exp['pv_train'].shape[1]} time steps (48 × 30 min)")


---
**Next:** see `02_train_and_evaluate.ipynb` for a complete training and inference walkthrough.
